# Sea Creature Image Classifier: A Guided Build

This notebook builds a deep-learning model that looks at a photo and names the sea
creature in it (dolphin, jellyfish, shark, whale, and so on). It is written as a set
of lecture notes: every step is explained before the code that carries it out, so you
can follow the reasoning and not just the syntax.

**What you will learn**

1. How image data is organised for classification, and how to explore it.
2. Why we resize, normalise, and augment images before training.
3. How to split data correctly so results are honest (no data leakage).
4. What transfer learning is, and why it beats training a network from scratch.
5. How a training loop actually works, line by line.
6. How to evaluate a model properly with a confusion matrix and per-class metrics.
7. How to interpret the model with Grad-CAM (seeing *where* it looks).

**The big picture.** Supervised image classification follows one recurring pipeline:

> raw images  ->  preprocess  ->  split  ->  model  ->  train  ->  evaluate  ->  predict

Keep that skeleton in mind. Every section below is one bone in it.

## 0. Environment setup

A quick word on the tools. We use **PyTorch** (the deep-learning framework),
**torchvision** (datasets, image transforms, and pretrained models),
**scikit-learn** (the data split and evaluation metrics), and **matplotlib** for plots.

If you are running this fresh (for example on Google Colab), uncomment the install
line in the next cell. If your environment already has these, skip it.

In [ ]:
# Uncomment on a fresh machine or Colab:
# !pip install torch torchvision scikit-learn matplotlib numpy Pillow tqdm kagglehub

In [2]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reproducibility: fixing the seed makes shuffles and splits repeatable, so your
# results do not change every time you re-run the notebook.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch", torch.__version__)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\User\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\User\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\User\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\User\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## 1. Getting the data

We use the **Sea Animals Image Dataset** by *vencerlanz09* on Kaggle:
`https://www.kaggle.com/datasets/vencerlanz09/sea-animals-image-dataste`

It contains roughly 13,000 images across 23 sea-creature classes. Please review and
respect the dataset's license on that page, and keep the attribution if you publish
this. The images are not ours to redistribute; we only train on them.

**The folder layout matters.** The dataset is stored in what torchvision calls
*ImageFolder* format: one subfolder per class, with that class's images inside.

```
sea_animals/
    Dolphin/     img001.jpg  img002.jpg  ...
    Jelly Fish/  img001.jpg  ...
    Sharks/      ...
    Whale/       ...
```

This convention is powerful because the folder *name* becomes the *label*. We never
have to write a separate list of labels; torchvision reads them from the directory
tree. Most classification datasets you meet will either come in this shape or be easy
to reshape into it.

**Two ways to get the files:**

- Automatic: the cell below uses `kagglehub`, which needs a free Kaggle API token
  (Kaggle > Settings > Create New Token).
- Manual: download the zip from the Kaggle page, unzip it, and point `DATA_DIR`
  (a few cells down) at the folder that directly contains the class subfolders.

In [ ]:
# Automatic download (needs a Kaggle token). Skip if you downloaded manually.
try:
    import kagglehub
    cache_path = kagglehub.dataset_download("vencerlanz09/sea-animals-image-dataste")
    print("Downloaded to:", cache_path)
except Exception as e:
    print("Automatic download skipped:", e)
    print("Download manually from Kaggle and set DATA_DIR in the next cell.")

Now we point `DATA_DIR` at the folder whose children are the class subfolders. If you
used `kagglehub` above, the images usually sit one level down inside `cache_path`. The
helper below finds the directory that contains the most subfolders, which is almost
always the class root.

In [ ]:
def find_class_root(path):
    """Return the directory whose children are the class folders."""
    best, best_count = path, sum(os.path.isdir(os.path.join(path, d)) for d in os.listdir(path))
    for root, dirs, _ in os.walk(path):
        count = sum(os.path.isdir(os.path.join(root, d)) for d in dirs)
        if count > best_count:
            best, best_count = root, count
    return best

# If you downloaded automatically, this resolves the class root for you.
# If you downloaded manually, replace the whole line with e.g.:
# DATA_DIR = "data/sea_animals"
DATA_DIR = find_class_root(cache_path) if "cache_path" in globals() else "data/sea_animals"
print("Using DATA_DIR =", DATA_DIR)

## 2. Exploring the data

Before modelling anything, look at it. This step is unglamorous and often skipped, and
that is exactly why so many projects quietly fail. Two questions to answer:

1. **How many images per class?** If some classes have far more images than others,
   the dataset is *imbalanced*, and a lazy model can score well just by guessing the
   common classes. We will need to account for that later.
2. **Do the images look sane?** A quick glance catches corrupt files, wrong labels,
   or surprises (a "shark" folder full of cartoon sharks would change everything).

In [ ]:
# Load the dataset once with no transform so we can inspect raw PIL images.
# ImageFolder assigns each class an integer index in alphabetical order.
raw = datasets.ImageFolder(DATA_DIR)
classes = raw.classes
num_classes = len(classes)

counts = np.bincount(raw.targets, minlength=num_classes)
print(f"{num_classes} classes, {len(raw)} images total\n")
for name, c in sorted(zip(classes, counts), key=lambda t: -t[1]):
    print(f"  {name:<18} {c}")

In [ ]:
# Visualise the class distribution. A tall spread between bars means imbalance.
order = np.argsort(counts)[::-1]
plt.figure(figsize=(11, 4))
plt.bar([classes[i] for i in order], counts[order])
plt.xticks(rotation=45, ha="right")
plt.ylabel("number of images")
plt.title("Images per class")
plt.tight_layout()
plt.show()

In [ ]:
# Peek at one random image from each of the first few classes as a sanity check.
sample_classes = classes[:min(8, num_classes)]
plt.figure(figsize=(13, 7))
for i, cname in enumerate(sample_classes):
    folder = os.path.join(DATA_DIR, cname)
    fname = random.choice(os.listdir(folder))
    img = Image.open(os.path.join(folder, fname)).convert("RGB")
    ax = plt.subplot(2, 4, i + 1)
    ax.imshow(img)
    ax.set_title(cname)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Transforms and data augmentation

A neural network cannot eat a JPEG. It needs a tensor of numbers with a fixed shape.
The job of a *transform* is to turn a raw image into that tensor. Our pipeline does
four things:

1. **Resize to 224x224.** The pretrained model we will use was trained on 224x224
   images, so we match that. Fixing the size also lets us stack images into batches.
2. **ToTensor.** Converts the image to a PyTorch tensor and rescales pixel values from
   the 0-255 integer range to a 0.0-1.0 float range.
3. **Normalize** with ImageNet's mean and standard deviation. Our backbone was trained
   on ImageNet with these exact statistics, so we shift and scale each colour channel
   the same way. This keeps the input distribution the model expects and helps it train
   smoothly.
4. **Augmentation (training only).** We randomly flip, rotate, and jitter the colours
   of *training* images. Each epoch the model sees slightly different versions of the
   same photo, which discourages it from memorising specific pixels and pushes it to
   learn the general shape of a creature. This is one of the cheapest ways to fight
   overfitting.

**Important asymmetry:** we augment only the training set. Validation and test images
get resized and normalised but never randomly altered, because we want to measure
performance on clean, realistic inputs.

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 4. Splitting the data: train, validation, test

We carve the data into three parts, each with a different job:

- **Train (about 70 percent):** the model learns from these.
- **Validation (about 15 percent):** we check on these *during* training to tune
  choices (how many epochs, which learning rate). The model never learns from them
  directly, but we use them to make decisions, so they are slightly "seen".
- **Test (about 15 percent):** touched only once, at the very end, to report an honest
  final score. Think of it as the exam you are not allowed to peek at.

**Data leakage** is the cardinal sin here. If information from validation or test
sneaks into training, your reported accuracy becomes a fantasy that collapses in the
real world. That is why augmentation is applied per split, and why the test set stays
sealed until the end.

**Stratified splitting.** A naive random split can, by chance, put most whales in the
test set and almost none in training. *Stratified* splitting keeps each class's
proportion roughly equal across all three splits, which matters a lot for imbalanced
data.

**One subtlety.** We loaded the dataset once with no transform. But train and eval need
*different* transforms. The `SplitDataset` wrapper below solves this: it holds a list
of indices and applies the right transform on the fly, so a single underlying dataset
can feed all three splits without leaking augmentation.

In [ ]:
targets = raw.targets  # integer label for every image, in dataset order

# First split off 30 percent for val+test, stratified by class.
train_idx, temp_idx = train_test_split(
    np.arange(len(raw)), test_size=0.30, stratify=targets, random_state=SEED
)
# Then split that 30 percent evenly into val and test (15 percent each).
temp_targets = [targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_targets, random_state=SEED
)

print(f"train {len(train_idx)}  |  val {len(val_idx)}  |  test {len(test_idx)}")


class SplitDataset(Dataset):
    """View a subset of `base` (by index) with a chosen transform."""
    def __init__(self, base, indices, transform):
        self.base = base
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        image, label = self.base[self.indices[i]]  # image is a PIL image
        return self.transform(image), label


train_ds = SplitDataset(raw, train_idx, train_tf)
val_ds = SplitDataset(raw, val_idx, eval_tf)
test_ds = SplitDataset(raw, test_idx, eval_tf)

## 5. DataLoaders: feeding the model in batches

We do not hand the model one image at a time, nor all of them at once. We feed
**batches** (here, 32 images). Batching is a compromise: big enough to use the GPU
efficiently and give a stable estimate of the gradient, small enough to fit in memory.

- **shuffle=True** for training so the model does not see classes in a fixed order
  (order can create spurious patterns). We leave validation and test unshuffled since
  order does not matter when we are only measuring.
- **num_workers** spins up background processes to load images while the GPU is busy.
  On Windows or in some notebooks this can misbehave, so set it to 0 if you hit errors.
- **pin_memory** speeds up the hand-off to a GPU; harmless on CPU.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2  # set to 0 if you get multiprocessing errors in a notebook

common = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **common)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **common)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **common)

# Per-class counts on the TRAIN split, used later for class-weighted loss.
train_targets = [targets[i] for i in train_idx]
train_counts = np.bincount(train_targets, minlength=num_classes)

# Sanity check: one batch should be [B, 3, 224, 224] images and [B] labels.
xb, yb = next(iter(train_loader))
print("batch images:", tuple(xb.shape), "| batch labels:", tuple(yb.shape))

## 6. The model: transfer learning

Here is the central idea of the whole project.

A convolutional neural network learns a hierarchy of visual features. Early layers
detect edges and colours; middle layers detect textures and simple shapes; late layers
detect complex parts (an eye, a fin, a shell). Training all of that from scratch needs
millions of images and a lot of compute, which we do not have.

**Transfer learning** borrows a network that already learned those features on a huge
dataset (ImageNet, 1.2 million images) and adapts it to our task. The insight is that
edges and textures are universal: the features useful for telling apart 1,000 everyday
objects are also useful for telling apart sea creatures. We keep the backbone's learned
features and only replace its final layer.

We use **ResNet18**, a compact, reliable network. Its last layer (`model.fc`) originally
outputs 1,000 ImageNet scores. We swap it for a fresh linear layer that outputs one
score per sea-creature class.

**Freeze or fine-tune?**

- *Freeze the backbone* (`freeze_backbone=True`): train only the new final layer. Fast,
  low memory, good when you have little data.
- *Fine-tune everything* (default here): let the whole network adjust to sea creatures.
  Usually the higher-accuracy choice when you have a few thousand images, as we do.

In [ ]:
def build_model(num_classes, freeze_backbone=False):
    weights = models.ResNet18_Weights.DEFAULT  # ImageNet-pretrained weights
    model = models.resnet18(weights=weights)

    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False

    # Replace the 1000-way ImageNet head with our own. New layers are trainable.
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


model = build_model(num_classes, freeze_backbone=False)
print(model.fc)  # confirm the head now outputs `num_classes` scores

## 7. Device, loss, optimizer, scheduler

Four ingredients turn a model into a trainable system.

**Device.** Training on a GPU is often 10 to 50 times faster than on a CPU. We pick a
CUDA GPU if present, then Apple Silicon (MPS), then fall back to CPU.

**Loss function: cross-entropy.** For classification, cross-entropy measures how far the
model's predicted probability distribution is from the true label. Minimising it pushes
the probability of the correct class toward 1.

**Class weights.** Because the dataset is imbalanced, we weight the loss so that mistakes
on rare classes cost more. This stops the model from ignoring the small classes to chase
overall accuracy. The weight for a class is inversely proportional to how many training
images it has.

**Optimizer: AdamW.** The optimizer is the rule for updating weights from gradients.
AdamW is a robust default that adapts the step size per parameter. `weight_decay` gently
shrinks weights toward zero, a form of regularisation that curbs overfitting.

**Learning-rate schedule.** We start at a modest learning rate and let a cosine schedule
ease it down over training. Large steps early explore quickly; small steps late settle
into a good minimum.

In [ ]:
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Using device:", device)
model = model.to(device)

# Inverse-frequency class weights (rarer class -> larger weight).
counts_f = train_counts.astype(np.float64)
weights = counts_f.sum() / (num_classes * np.clip(counts_f, 1, None))
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 10          # raise to 15-20 for better results if you have a GPU
LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 8. The training loop

This is where learning happens. One **epoch** is one full pass over the training data.
Within an epoch, for each batch, the same five steps repeat:

1. **Forward pass:** run the images through the model to get predicted scores.
2. **Compute the loss:** compare predictions to the true labels.
3. **Backward pass:** `loss.backward()` computes the gradient of the loss with respect
   to every weight (how each weight should change to reduce the loss).
4. **Update:** `optimizer.step()` nudges the weights along those gradients.
5. **Reset:** `optimizer.zero_grad()` clears the gradients so the next batch starts
   clean. PyTorch accumulates gradients by default, so forgetting this is a classic bug.

Two modes matter. `model.train()` enables training behaviours (dropout, batch-norm
updates); `model.eval()` with `torch.no_grad()` disables them and skips gradient
tracking, which is both correct and faster for validation.

After each epoch we check the validation accuracy and **save the model only when it
improves**. This "keep the best checkpoint" habit means a late epoch that overfits will
not overwrite our best model.

In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    """One pass over `loader`. Trains if `optimizer` is given, else evaluates."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, seen = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()
            logits = model(images)          # 1. forward
            loss = criterion(logits, labels)  # 2. loss
            if training:
                loss.backward()             # 3. backward
                optimizer.step()            # 4. update

            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            seen += images.size(0)

    return total_loss / seen, correct / seen

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
CKPT_PATH = "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, device, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(
            {"state_dict": model.state_dict(), "classes": classes, "img_size": IMG_SIZE},
            CKPT_PATH,
        )
        print(f"  new best (val acc {va_acc:.3f}) saved to {CKPT_PATH}")

print(f"\nBest validation accuracy: {best_val_acc:.3f}")

## 9. Reading the training curves

Plotting loss and accuracy over epochs tells a story:

- **Healthy learning:** both training and validation loss fall and level off.
- **Overfitting:** training loss keeps falling while validation loss turns back upward.
  The model is memorising the training set instead of generalising. Fixes: more
  augmentation, more weight decay, fewer epochs, or more data.
- **Underfitting:** both losses stay high. The model is too weak or under-trained.
  Fixes: train longer, unfreeze the backbone, raise the learning rate.

The gap between the two curves is the thing to watch. A small gap is what we want.

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs_range, history["train_loss"], label="train")
ax1.plot(epochs_range, history["val_loss"], label="val")
ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()
ax2.plot(epochs_range, history["train_acc"], label="train")
ax2.plot(epochs_range, history["val_acc"], label="val")
ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout()
plt.show()

## 10. Evaluation on the test set

Now we open the sealed exam. We reload the *best* checkpoint (not necessarily the last
epoch's model) and score it on the test split it has never influenced.

**Why not just accuracy?** Overall accuracy hides per-class behaviour. On imbalanced
data a model can score high overall while failing a rare class entirely. So we report:

- **Precision:** of the images the model called class X, how many really were X.
- **Recall:** of the images that truly are X, how many the model found.
- **F1:** the harmonic mean of precision and recall, a single balanced number.
- **Macro average:** the unweighted mean across classes, which treats a rare class as
  seriously as a common one. Watch this number for imbalanced data.

**The confusion matrix** is the richest view. Row = true class, column = predicted
class. A perfect model lights up only the diagonal. Bright off-diagonal cells reveal
*which* creatures get mistaken for which (dolphins and sharks, perhaps), which is far
more actionable than a single accuracy figure.

In [ ]:
# Reload the best checkpoint into a fresh model.
ckpt = torch.load(CKPT_PATH, map_location=device)
best_model = build_model(len(ckpt["classes"]))
best_model.load_state_dict(ckpt["state_dict"])
best_model.to(device).eval()

# Collect predictions over the whole test set.
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = best_model(images.to(device)).argmax(1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.numpy().tolist())

print(classification_report(y_true, y_pred, target_names=classes, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=True
)
ax.set_title("Confusion matrix (test split)")
plt.tight_layout()
plt.show()

## 11. Predicting a single image

A trained model is only useful if you can point it at a new photo. Inference is short:
apply the same *evaluation* transform, add a batch dimension, run a forward pass, and
turn the raw scores into probabilities with **softmax**. We then report the top few
guesses with their confidences.

In [ ]:
def predict_image(path, model, classes, topk=3):
    image = Image.open(path).convert("RGB")
    x = eval_tf(image).unsqueeze(0).to(device)  # shape [1, 3, 224, 224]
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1).squeeze(0)
    top_p, top_i = probs.topk(min(topk, len(classes)))
    results = [(classes[i], float(p)) for p, i in zip(top_p, top_i)]
    return image, results


# Demo on one random test image so the notebook runs end to end.
demo_class = random.choice(classes)
demo_path = os.path.join(
    DATA_DIR, demo_class, random.choice(os.listdir(os.path.join(DATA_DIR, demo_class)))
)
img, results = predict_image(demo_path, best_model, classes)

plt.imshow(img); plt.axis("off")
plt.title("true: " + demo_class)
plt.show()
for label, p in results:
    print(f"  {label:<18} {p * 100:5.1f}%")

## 12. Bonus: Grad-CAM, seeing what the model looks at

A model can be right for the wrong reasons (predicting "shark" from the blue water
rather than the animal). **Grad-CAM** (Gradient-weighted Class Activation Mapping) opens
the black box a little. It produces a heatmap over the image showing which regions most
influenced a given prediction.

The intuition: take the last convolutional layer's feature maps, weight each map by how
much it pushed up the score for the predicted class (obtained from gradients), sum them,
and keep the positive parts. Bright areas are where the model "looked". If the heat lands
on the creature, the model is reasoning about the right thing. If it lands on the
background, be suspicious.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.eval()
        logits = self.model(x)
        if class_idx is None:
            class_idx = int(logits.argmax(1))
        self.model.zero_grad()
        logits[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)     # one weight per channel
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)     # normalise to 0..1
        return cam, class_idx


# The last residual block of ResNet18 holds the richest spatial features.
cam_engine = GradCAM(best_model, best_model.layer4[-1])

image = Image.open(demo_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
x = eval_tf(Image.open(demo_path).convert("RGB")).unsqueeze(0).to(device)
heatmap, idx = cam_engine(x)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(image); axes[0].set_title("input"); axes[0].axis("off")
axes[1].imshow(image)
axes[1].imshow(heatmap, cmap="jet", alpha=0.5)
axes[1].set_title(f"Grad-CAM: {classes[idx]}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 13. Saving the model for reuse

The checkpoint we saved during training holds three things: the learned weights, the
class-name list (so predictions map back to readable labels), and the image size. That
bundle is everything a separate app or a Gradio demo needs to load the model and make
predictions, with no need to retrain.

In [ ]:
torch.save(
    {"state_dict": best_model.state_dict(), "classes": classes, "img_size": IMG_SIZE},
    "sea_creature_model.pt",
)
print("Saved sea_creature_model.pt")
print("Load it elsewhere with torch.load(...) and the build_model function above.")

## 14. Recap and where to go next

**What we did, in one breath:** loaded an ImageFolder dataset, explored its class
balance, preprocessed and augmented the images, split them without leakage, fine-tuned a
pretrained ResNet, trained it with a class-weighted loss, evaluated it honestly with a
confusion matrix and per-class metrics, ran single-image inference, and used Grad-CAM to
check the model looks at the right pixels.

**Ideas to extend it** (each is a good portfolio talking point):

- Try a stronger backbone (`resnet34`, `resnet50`, or an EfficientNet) and compare.
- Add test-time augmentation for a small, cheap accuracy gain.
- Address the hardest confusions you found in the matrix (gather more images, or merge
  genuinely ambiguous classes).
- Wrap the saved model in a Gradio app and host it free on Hugging Face Spaces, then
  embed the live demo in your portfolio site.
- Export to TorchScript or ONNX for faster, framework-independent deployment.

**For your writeup:** state the final test accuracy and macro F1, drop in the training
curve and confusion-matrix images, and add a short paragraph on what surprised you.
That reflection is often what a reviewer remembers.